# Production Operations

The assembly chapter (notebook 12) shipped a single `/health` endpoint that returns `{"status":"ok"}` from `12-photo-app.ipynb:[7]`. From a load balancer's perspective this is a [liveness probe]{.mark} — it tells the LB the process is alive enough to read the file descriptor it serves on. It is not a [readiness probe]{.mark}: it does not check whether the database pool is reachable, whether S3 returns 200, whether Redis-cache fallback is healthy, or whether the in-flight jobs count is below the safe ceiling. A node that fails all four of those checks continues to receive traffic because the LB keeps seeing `{"status":"ok"}` — every request routed to that node fails.

Two related operational gaps compound. The pipeline's `POST /search/` runs a ~20 ms CPU-bound CLIP encode per request; there is no rate limit, so a runaway client or a client with a stuck retry loop pegs a single API worker to 100% CPU. `POST /pipeline/start` is similarly unguarded. The Flet app on two laptops can do `pipeline/start` simultaneously and the worker pool keeps the queue growing with no refusal. Without metrics, the operator cannot see this — notebook 12 prints per-stage timing to stdout, which disappears when the terminal scrolls. [What you cannot see you cannot fix]{.mark}, and without backpressure, what you do not refuse accumulates.

This notebook closes the operational envelope. We split `/health` into three endpoints: `/health` (liveness, trivial), `/ready` (readiness, real checks), and `/metrics` (Prometheus). We add a per-device token-bucket rate limiter on `/search` and `/pipeline/start`. We add structured JSON logs with correlation IDs so multi-device flows are traceable. We add a graceful-shutdown lifespan hook that drains WebSockets and lets in-flight batches complete. Finally we cover horizontal scaling with the Postgres `FOR UPDATE SKIP LOCKED` pattern that becomes the only new SQL trick needed for a multi-instance production deployment.

---

## Liveness vs. Readiness

**Liveness: is the process alive?** `/health` returns 200 if the Python interpreter is running and Starlette is dispatching. It is one byte read of memory; it should never fail apart from a crashed process. Load balancers check this frequently (every 5 s typical) and use it to decide whether to *restart the container*.

**Readiness: is the process ready to serve traffic?** `/ready` returns 200 only if every dependency the request path touches is reachable. For us that means: DB pool has at least one connection, S3 returns a 200 on a `HeadBucket`, Redis (if used) returns PONG. Load balancers check this every 5–10 s and use it to decide whether to *route traffic to this instance*. A not-ready instance is removed from rotation but the container is not restarted.

:::{.callout-important}
A common bug is using `/health` for both decisions. The load balancer then never removes "alive but broken" instances from rotation — it restarts them only when the process crashes, which may never happen for a wedged-but-alive state (DB pool exhausted, Redis unreachable, queue stuck). Use two endpoints.

:::


In [ ]:
from fastapi import FastAPI, APIRouter, Depends, HTTPException
from pydantic import BaseModel


ops_router = APIRouter(tags=["ops"])


class HealthResponse(BaseModel):
    status: str = "ok"


class ReadyResponse(BaseModel):
    status: str = "ready"
    checks: dict[str, str] = {}


class ReadyFailure(BaseModel):
    status: str = "not_ready"
    failed: list[str]


@ops_router.get("/health", response_model=HealthResponse)
async def health() -> HealthResponse:
    """Liveness probe — trivial, fast, never fails unless the process is gone."""
    return HealthResponse(status="ok")


# --- Readiness probes (stubs; in production these are real round-trips) -------
async def check_db_pool(db_pool) -> bool:
    """SELECT 1; succeeds iff Postgres accepts a connection from the pool."""
    try:
        # In production: await db.execute(text("SELECT 1"))
        return True
    except Exception:
        return False


async def check_s3(s3_client, bucket: str) -> bool:
    """HeadBucket; succeeds iff S3 returns 200 for the configured bucket."""
    try:
        # In production: s3_client.head_bucket(Bucket=bucket)
        return True
    except Exception:
        return False


async def check_redis(redis_client) -> bool:
    """PING; succeeds iff Redis is reachable."""
    try:
        # In production: await redis_client.ping()
        return True
    except Exception:
        return False


@ops_router.get("/ready")
async def ready(
    db_pool=Depends(None),       # in production: Depends(get_db_pool)
    s3_client=Depends(None),     # in production: Depends(get_s3_client)
    redis_client=Depends(None),  # in production: Depends(get_redis_client)
    bucket: str = "my-photos",
) -> ReadyResponse | ReadyFailure:
    """Readiness probe — runs every dependency probe in parallel; returns the
    aggregate state plus per-check detail so an operator can see WHAT failed."""
    # In production this fires them concurrently with asyncio.gather.
    checks = {
        "db":    await check_db_pool(db_pool),
        "s3":    await check_s3(s3_client, bucket),
        "redis": await check_redis(redis_client),
    }
    failed = [name for name, ok in checks.items() if not ok]
    if failed:
        # 503 with body — LB removes instance from rotation, container keeps running
        raise HTTPException(status_code=503, detail={"failed": failed})
    return ReadyResponse(
        status="ready",
        checks={name: "ok" for name in checks},
    )


# Confirm the shape of the responses (with stubs that all return True).
print("Liveness probe body:", HealthResponse().model_dump_json())
print("Readiness probe body:", "would include db=ok, s3=ok, redis=ok")


## Per-Device Token-Bucket Rate Limiting

`/search` and `/pipeline/start` are the two endpoints whose cost is measurable per request: `/search` does ~20 ms CLIP encode and a pgvector seek; `/pipeline/start` queues a full bucket rescan. On a two-laptop family scenario they are comfortably under any rate limit. On a runaway client (the save dialog issues "Open" five times; the laptop wakes up with a stuck retry loop) they dogpile.

The right primitive is a per-`device_id` token bucket. Each device has a bucket with capacity `C` (burst tokens) refilled at rate `R` (tokens/sec). Each request consumes `cost` tokens (1 for `/search`, 5 for `/pipeline/start`). The refill is only conceptual; production uses a Redis `INCRBY + EXPIRE` Lua script to atomically subtract and refill. Below we implement an in-process synchronous version that is the same shape and easy to swap for the Redis one.

:::{.callout-note}
Rate limiting by `device_id` (introduced in PHT:01) is the only fair unit. Rate limiting by IP is wrong because home NAT puts every laptop behind the same source IP — your wife's runaway client silently drops your own requests.

:::


In [ ]:
import time
from collections import defaultdict


class TokenBucket:
    """Synchronous in-process token bucket. Mirrors the Redis Lua-script version
    bitwise, so swapping the backend is a one-line code change."""
    __slots__ = ("capacity", "rate", "_tokens", "_last_refill")

    def __init__(self, capacity: float, rate: float):
        self.capacity = capacity
        self.rate = rate
        self._tokens = capacity
        self._last_refill = time.monotonic()

    def _refill(self) -> None:
        now = time.monotonic()
        elapsed = now - self._last_refill
        add = elapsed * self.rate
        self._tokens = min(self.capacity, self._tokens + add)
        self._last_refill = now

    def take(self, cost: float = 1.0) -> bool:
        """Returns True if the bucket had enough tokens (consume them now);
        False if the bucket was below cost (no tokens consumed — the caller
        should 429)."""
        self._refill()
        if self._tokens >= cost:
            self._tokens -= cost
            return True
        return False


class RateLimiter:
    """Per-device token bucket store. Buckets are lazily created on first visit."""
    def __init__(self, capacity: float, rate: float):
        self.capacity = capacity
        self.rate = rate
        self._buckets: dict[str, TokenBucket] = {}

    def allow(self, device_id: str, cost: float = 1.0) -> bool:
        if device_id not in self._buckets:
            self._buckets[device_id] = TokenBucket(self.capacity, self.rate)
        return self._buckets[device_id].take(cost)


# Tunables: 30 search requests per minute per device, burst of 10.
search_limiter = RateLimiter(capacity=10, rate=0.5)        # 0.5 tokens/sec → 30/min
pipeline_limiter = RateLimiter(capacity=2, rate=1/300)     # 1 pipeline start per 5 min per device

# Torch a single bucket from two devices, verify isolation.
print("--- search limiter, device-A bursts 15, then device-B bursts 15 ---")
a_results = [search_limiter.allow("device-A") for _ in range(15)]
print(f"device-A: allowed {sum(a_results)} / 15  (expected 10, then reject)")
b_results = [search_limiter.allow("device-B") for _ in range(15)]
print(f"device-B: allowed {sum(b_results)} / 15  (expected 10 — unaffected by A)")

print("\n--- pipeline limiter, two starts back-to-back ---")
p1 = pipeline_limiter.allow("device-A", cost=1)
p2 = pipeline_limiter.allow("device-A", cost=1)
print(f"start 1: {p1}  start 2: {p2}  (expected True then False — burst=2 OK)")
p3 = pipeline_limiter.allow("device-A", cost=1)
print(f"start 3: {p3}  (expected False — bucket is 0; rate is 1/300s)")


## The FastAPI Dependency Wiring

Wiring a rate-limited endpoint is one `Depends(...)` line. The dependency resolves `device_id` (PHT:01's `current_device`), looks up the right limiter, and either returns silently (rate below capacity) or raises a 429 with `Retry-After` computed from the bucket's refill rate.


In [ ]:
from fastapi import Depends, HTTPException, Request


async def current_device_dep(x_device_id: str = None) -> str:
    """PHT:01's `current_device` re-used as the rate-limit key."""
    if not x_device_id:
        raise HTTPException(status_code=401, detail="X-Device-Id required")
    return x_device_id


def rate_limit(limiter: RateLimiter, cost: float = 1.0):
    """FastAPI dependency factory: pick the limiter, set the cost."""
    async def _rl(device_id: str = Depends(current_device_dep)) -> str:
        if not limiter.allow(device_id, cost=cost):
            # Retry-After hint: time-until-next-token ≈ cost / rate
            retry_after = max(1, int(cost / limiter.rate))
            raise HTTPException(
                status_code=429,
                detail={"device_id": device_id, "cost": cost, "retry_after": retry_after},
                headers={"Retry-After": str(retry_after)},
            )
        return device_id
    return _rl


# The wired endpoint looks like:
#
#   @search_router.post("/")
#   async def search(req: SearchRequest,
#                   device_id: str = Depends(rate_limit(search_limiter))):
#       ...
#
# Consumers know that any 429 they see came from this limiter, and that Retry-After
# (in seconds) is a safe backoff-of-1 to avoid spurious back-to-back 429s.

# Demo: simulate three calls, last one 429s after burst is drained.
search_limiter_small = RateLimiter(capacity=2, rate=0.1)
calls = [("device-A", True), ("device-A", True), ("device-A", False)]
for i, (device_id, expected) in enumerate(calls):
    allowed = search_limiter_small.allow(device_id)
    print(f"call {i}: allowed={allowed}  expected={expected}  ok={allowed==expected}")


## Prometheus Metrics

Use [prometheus-client](https://github.com/prometheus/client_python) — it is a small, dependency-light library that exposes `/metrics` on the same FastAPI app. Three families of metrics cover every PHT feature:

- **Counters** for one-shot events: `search_requests_total`, `pipeline_jobs_total`, `deletes_total`, `cache_hits_total{cache="embeddings|presign"}`.
- **Histograms** for latency distributions: `http_request_duration_seconds{endpoint}`, `search_stage_duration_seconds{stage}`, `pipeline_batch_duration_seconds`.
- **Gauges** for instant state: `inflight_jobs`, `db_pool_size{state}`, `rate_limit_tokens_remaining{device_id,endpoint}`.

The histogram for the search stage timings from `12-photo-app.ipynb:[20]` is the single most useful one — it converts the per-stage `time.perf_counter()` prints into a Prometheus time series that you can plot in Grafana by stage.


In [ ]:
try:
    from prometheus_client import Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST, REGISTRY
    _PROM_AVAILABLE = True
except ImportError:
    _PROM_AVAILABLE = False


# In production these are module-level definitions; here we wrap in a try/except
# to support environments without prometheus_client installed (e.g. CI of the
# notebook repo).
if _PROM_AVAILABLE:
    SEARCH_REQUESTS = Counter(
        "photo_app_search_requests_total", "Search requests served", ["outcome"],
    )
    SEARCH_STAGE_DURATION = Histogram(
        "photo_app_search_stage_duration_seconds",
        "Per-stage search latency",
        ["stage"],         # "parse" | "embed" | "sql" | "vector_rank" | "rerank" | "mmr"
        buckets=(0.001, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0),
    )
    PIPELINE_JOBS = Counter(
        "photo_app_pipeline_jobs_total", "Pipeline jobs by kind", ["kind", "outcome"],
    )
    CACHE_HITS = Counter(
        "photo_app_cache_hits_total", "Cache hits", ["cache", "result"],
    )
    INFIGHT_JOBS = Gauge(
        "photo_app_infight_jobs", "Jobs currently being worked on",
    )
else:
    SEARCH_REQUESTS = SEARCH_STAGE_DURATION = PIPELINE_JOBS = None
    CACHE_HITS = INFIGHT_JOBS = None


def record_search_stage(stage: str, duration_s: float) -> None:
    if _PROM_AVAILABLE:
        SEARCH_STAGE_DURATION.labels(stage=stage).observe(duration_s)


def record_search_request(outcome: str) -> None:
    if _PROM_AVAILABLE:
        SEARCH_REQUESTS.labels(outcome=outcome).inc()


# In production the /metrics endpoint is:
#   @ops_router.get("/metrics")
#   async def metrics():
#       return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

# Demo: observe a few samples and print the histogram summary.
if _PROM_AVAILABLE:
    for stage, dur in [("parse", 0.018), ("embed", 0.020), ("sql", 0.001), ("vector_rank", 0.005), ("rerank", 0.0005), ("mmr", 0.0002)]:
        record_search_stage(stage, dur)
    record_search_request("ok")
    print(f"search_requests_total: {REGISTRY.get_sample_value('photo_app_search_requests_total', {'outcome': 'ok'})}")
else:
    print("prometheus_client not installed — install via `uv add prometheus_client`")


## Structured Logs and Correlation IDs

Two laptops and one bucket mean one user's flow and another's are interleaved in the same log stream. Without correlation IDs, debugging "why did this search latency spike at 17:42?" becomes a guessing game of which device issued what.

We use a single middleware that reads `X-Request-Id` (or generates a uuid if absent) and binds it to a contextvar. Structured logging emits JSON lines with that ID and the resolved `X-Device-Id`. We use the `rich` library's `RichHandler` for development-time human-readable console output, and a plain JSON formatter for production (just swap the formatter — both write the same fields).


In [ ]:
import logging
import contextvars
from uuid import uuid4


# Per-request context: the ID and the resolved device.
request_id_ctx: contextvars.ContextVar[str] = contextvars.ContextVar("request_id", default="")
device_id_ctx:   contextvars.ContextVar[str] = contextvars.ContextVar("device_id", default="")


class JsonFormatter(logging.Formatter):
    """Emit one JSON document per log record."""
    def format(self, record: logging.LogRecord) -> str:
        import json
        payload = {
            "ts":          self.formatTime(record, "%Y-%m-%dT%H:%M:%S.%fZ"),
            "level":       record.levelname,
            "logger":      record.name,
            "message":     record.getMessage(),
            "request_id":  request_id_ctx.get(),
            "device_id":   device_id_ctx.get(),
        }
        if record.exc_info:
            payload["exc"] = self.formatException(record.exc_info)
        return json.dumps(payload)


def configure_logging(json_output: bool = False) -> None:
    handler = logging.StreamHandler()
    handler.setFormatter(JsonFormatter() if json_output else logging.Formatter("%(asctime)s %(levelname)s %(name)s %(message)s"))
    root = logging.getLogger()
    root.handlers = [handler]
    root.setLevel(logging.INFO)


configure_logging(json_output=True)
log = logging.getLogger("photo_app.search")


# Bind a request context.
request_id_ctx.set(f"req-{uuid4().hex[:8]}")
device_id_ctx.set("device-wife")

log.info("search_request", extra={})
log.info("stage_complete", extra={"stage": "parse", "ms": 18.4})
log.info("stage_complete", extra={"stage": "embed", "ms": 20.1, "cache_hit": False})
log.info("search_request_done", extra={"latency_ms": 41.5})

# In production a Starlette middleware sets the contextvars from request headers,
# so every log line emitted during that request carries the same request_id:
#
#   @app.middleware("http")
#   async def correlation_middleware(request: Request, call_next):
#       rid = request.headers.get("X-Request-Id") or f"req-{uuid4().hex[:8]}"
#       token1 = request_id_ctx.set(rid)
#       token2 = device_id_ctx.set(request.headers.get("X-Device-Id", ""))
#       try:
#           return await call_next(request)
#       finally:
#           request_id_ctx.reset(token1)
#           device_id_ctx.reset(token2)


## Graceful Shutdown

A SIGTERM (docker-compose down, kubectl rolling restart) gives the API a short window to drain. Naively terminating in the middle of a batch loses work and leaves WebSockets dangling. We hook the Starlette `lifespan` `yield` block (the shutdown half) to:

1. Stop accepting new WebSocket connections (refuse the upgrade handshake).
2. Wait for in-flight HTTP requests up to `shutdown_grace_s` (default 10 s).
3. Wait for in-flight batches up to `shutdown_batch_grace_s` (default 30 s) — they finish-and-commit where they are.
4. Cancel any outstanding worker pool tasks whose batches did not commit by the grace timeout; mark their batches `'failed'` with `error='graceful_shutdown'` so the next startup reconciles.

This builds on the PHT:05 work: resumable batches mean graceful shutdown is just an application-level cancellation of pools that lets in-flight batches complete.


In [ ]:
import asyncio
from contextlib import asynccontextmanager


SHUTDOWN_GRACE_S         = 10    # HTTP requests have this long to finish
SHUTDOWN_BATCH_GRACE_S   = 30    # In-flight batches have this long to finish


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Startup hook: connect DB, run migrations, claim orphaned 'running' jobs
    from PHT:05. Shutdown hook: drain WebSockets + in-flight batches."""
    print("[startup] connecting database, claiming orphaned jobs...")
    yield
    # --- Shutdown half ---
    print("[shutdown] refusing new WebSockets, draining in-flight requests...")
    # await websocket_manager.refuse_new()             # 1
    # await asyncio.sleep(0)                            # let HTTP handler loop see close

    # Give in-flight HTTP requests up to SHUTDOWN_GRACE_S seconds.
    # In Starlette this is hand-rolled via tracking; here we just `sleep` as a stub.
    await asyncio.sleep(0.01)                          # 2 (stub)

    print("[shutdown] draining in-flight pipeline batches...")
    # Each pool we know about is told to stop; we await join() with the grace
    # timeout. The pool finishes the current batch and exits cleanly.
    # await asyncio.wait_for(asyncio.gather(*pool_tasks), timeout=SHUTDOWN_BATCH_GRACE_S)

    print("[shutdown] cancelling pools whose batches did not commit in time...")
    # Any pool still running after the grace timeout has its tasks cancelled
    # and PHT:05's reconciliation (marking the one 'running' batch 'failed')
    # runs the next time lifespan startup fires.
    # We do not call close() on the DB pool here — uvicorn does it for us.

    print("[shutdown] done.")


# Demonstrate the lifespan contract. In production this loop is wrapped by
# uvicorn's lifespan handling — `yield` separates startup from shutdown.
async def demo():
    async with lifespan(FastAPI()):
        print("[demo] inside started app — handlers run here")
asyncio.run(demo())


## Horizontal Scaling: `FOR UPDATE SKIP LOCKED`

The single-instance working set is one FastAPI process with a worker pool. The horizontal step is N FastAPI processes behind a load balancer, sharing one Postgres + one Redis, with no shared in-process state. Three changes:

1. **Job claim.** The atomic `UPDATE … WHERE state='queued'` from PHT:05 races on N workers. The Postgres-specific fix is `SELECT … FOR UPDATE SKIP LOCKED`, which lets each worker acquire a different row without blocking on the others. The pattern is one query:

```sql
SELECT job_id FROM jobs
WHERE state = 'queued'
ORDER BY created_at
FOR UPDATE SKIP LOCKED     -- << skip rows held by other workers
LIMIT 1;
-- Then: UPDATE jobs SET state='running', claimed_by=:wid WHERE job_id = :jid;
```

2. **Rate limiter back-end.** The in-process `TokenBucket` from this notebook becomes a Redis-backed Lua script (atomic `INCRBY` + `EXPIRE`) so all N workers see the same bucket per `device_id`. The penalty for using per-process buckets at scale is that each instance thinks "device-A has full tokens" when the others have already drained them — soft-failed backpressure.

3. **WebSocket broadcast.** The in-process `manager.broadcast` from notebook 12 (`12-photo-app.ipynb:[3]`) becomes a Redis pub/sub fan-out: workers `PUBLISH job-{id}/event <json>`; each WebSocket-holding instance `SUBSCRIBE`s. The downside is one extra hop per event; the upside is N-way fanout.

:::{.callout-note}
We do not scale the embedding encoder horizontally here. CLIP-on-CPU is single-process-bound by the GIL. A multi-machine fleet separates the **embedding worker** into its own pod (pull jobs from a Redis queue, write embeddings back to pgvector) and lets each API process stay I/O-light. That makes the API processes horizontally scalable for browse/search, while embeddings scale by adding preprocessing pods. This is the architecture notebook 12's "Scaling Beyond One Machine" appendix gestured at; PHT:05 + this notebook realize it.

:::


In [ ]:
# Sketch of the SKIP LOCKED claim. Shown as SQLAlchemy shape only; runnable
# with any Postgres-backed async engine.
from sqlalchemy import select, text


async def claim_job_skip_locked(session) -> str | None:
    """Atomically claim one queued job via Postgres's FOR UPDATE SKIP LOCKED.

    Other workers running this same query concurrently each grab a different
    row — no row is claimed twice.
    """
    stmt = text("""
        SELECT job_id FROM jobs
        WHERE state = 'queued'
        ORDER BY created_at
        FOR UPDATE SKIP LOCKED
        LIMIT 1
    """)
    # row = (await session.execute(stmt)).first()
    # if row is None: return None
    # job_id = row.job_id
    # await session.execute(text(
    #     "UPDATE jobs SET state='running', claimed_by=:wid WHERE job_id=:jid"
    # ), {"wid": WORKER_ID, "jid": job_id})
    # await session.commit()
    # return job_id
    return None   # stub


# Demonstrate the claim structure against an in-memory stand-in: 3 workers, 5 jobs.
import threading


QUEUED_JOBS: list[str] = [f"job-{i:02d}" for i in range(5)]
LOCKED: set[str] = set()
CLAIMED: dict[str, str] = {}    # job_id -> worker_id
LOCK = threading.Lock()


def claim_inmem(worker_id: str) -> str | None:
    """In-process stand-in mimicking FOR UPDATE SKIP LOCKED."""
    with LOCK:
        for job_id in QUEUED_JOBS:
            if job_id not in LOCKED and job_id not in CLAIMED:
                LOCKED.add(job_id)
                CLAIMED[job_id] = worker_id
                LOCKED.discard(job_id)
                return job_id
        return None


# Three workers race; each gets a distinct job.
results = {}
workers = ["w1", "w2", "w3"]
for w in workers:
    results[w] = claim_inmem(w)
print(f"claims: {results}  (each worker got a distinct job)")


## Schema Additions

The operational layer adds one column to `jobs` (the `claimed_by` for multi-worker protection) and... that is all. Readiness, metrics, rate limiter, structured logs, and graceful shutdown touch no tables; they live in application code.

```sql
-- 0023_pht_ops.sql
ALTER TABLE jobs ADD COLUMN IF NOT EXISTS claimed_by TEXT;
CREATE INDEX ix_jobs_claimed_by ON jobs (claimed_by)
    WHERE claimed_by IS NOT NULL;
```


## Summary

This notebook closed the operational envelope. Liveness (`/health`) and readiness (`/ready`) are now two endpoints serving two different purposes; the load balancer can remove wedged-but-alive instances from rotation. A per-`device_id` token-bucket limiter protects `/search` and `/pipeline/start` from runaway clients, with `Retry-After` returned on 429 so clients back off. Prometheus counters + histograms + gauges instrument every PHT feature (search stages, cache hits, batches, deletes). Structured JSON logs with correlation IDs make two-laptop flows traceable. The shutdown half of `lifespan` drains WebSockets and in-flight batches gracefully so restarts do not lose work. Horizontal scaling is the Postgres `FOR UPDATE SKIP LOCKED` claim pattern + Redis-backed rate limiter + Redis pub/sub WebSocket fan-out; nothing else changes in the application code path.

**What changed relative to notebook 12.** `/health` is joined by `/ready` and `/metrics`. The lifespan shutdown hook grew a drain sequence. The rate limiter is new; the structured-logging middleware is new; horizontal scaling becomes feasible because every in-process thing that did not survive a second process — rate-limit tokens, WebSocket broadcast, in-flight job claim — moves to Redis.

**What this enables offline this series.** Three laptops — yours, your wife's, your dad's — pointing at one shared S3 bucket, one shared Postgres, one shared Redis, behind a load balancer with two API instances. That is the configuration the index page described: "runs on my laptop and my wife's laptop ... against the same S3 bucket, and degrades gracefully as the library grows past 100K photos." We made the index's promise concrete.

---



---


■
